In [ ]:
!pip install -U langchain langchain-core langchain-classic -q

In [ ]:
!pip install -U langchain-openai -q

In [1]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

# Define Tools

In [2]:
@tool
def calculator(expression: str) -> str:
    """Evaluate a basic math expression. Example: '2 + 2 * 5'"""
    try:
        result = eval(expression, {"__builtins__": {}})
        return str(result)
    except Exception as e:
        return f"Error: {e}"


@tool
def word_counter(text: str) -> str:
    """Count the number of words in a given text."""
    count = len(text.split())
    return f"{count} words"


# Build the Agent

In [8]:
import os
os.environ["OPENAI_API_KEY"] = "sk-..." # Give your OpenAI API key here (if you run this locally then You need to set the OPENAI_API_KEY environment variable before ChatOpenAI() is instantiated)

In [9]:
def build_agent():
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0,
        api_key=os.environ.get("OPENAI_API_KEY"),
    )

    tools = [calculator, word_counter]

    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant. Use the tools available to answer questions accurately."),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ])

    agent = create_openai_tools_agent(llm, tools, prompt)
    return AgentExecutor(agent=agent, tools=tools, verbose=True)
    



# Run the agent

In [12]:
import os
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.agents import AgentExecutor, create_openai_tools_agent

if __name__ == "__main__":
    agent_executor = build_agent()

    queries = [
        "What is (42 * 7) + 18?",
        "How many words are in: 'The quick brown fox jumps over the lazy dog'?",
        "Calculate 100 / 4, then tell me if that's more or less than 30.",
    ]

    for query in queries:
        print(f"\n{'='*50}")
        print(f"Query: {query}")
        result = agent_executor.invoke({"input": query})
        print(f"Answer: {result['output']}")


Query: What is (42 * 7) + 18?


> Entering new AgentExecutor chain...

Invoking: `calculator` with `{'expression': '(42 * 7) + 18'}`


312The result of the expression \((42 * 7) + 18\) is 312.

> Finished chain.
Answer: The result of the expression \((42 * 7) + 18\) is 312.

Query: How many words are in: 'The quick brown fox jumps over the lazy dog'?


> Entering new AgentExecutor chain...

Invoking: `word_counter` with `{'text': 'The quick brown fox jumps over the lazy dog'}`


9 wordsThe phrase "The quick brown fox jumps over the lazy dog" contains 9 words.

> Finished chain.
Answer: The phrase "The quick brown fox jumps over the lazy dog" contains 9 words.

Query: Calculate 100 / 4, then tell me if that's more or less than 30.


> Entering new AgentExecutor chain...

Invoking: `calculator` with `{'expression': '100 / 4'}`


25.0The result of \( 100 / 4 \) is 25. Since 25 is less than 30, the answer is less than 30.

> Finished chain.
Answer: The result of \( 100 / 4 \) is 25. Since